In [ ]:
import os, sys, json
sys.path.append('..')
from utils.helpers import get_client, CLAUDE_HAIKU, DEFAULT_MODEL

client = get_client()
MODEL = DEFAULT_MODEL
MODEL_JUDGE = CLAUDE_HAIKU

print("Celda de configuración OK")


## 1. Generando el Dataset de Prueba

In [ ]:
# Dataset: tarea de clasificación de sentimientos
# Formato: (input, expected_label)

DATASET = [
    # Positivos
    ("¡Este producto es increíble! Lo recomiendo totalmente.",   "positivo"),
    ("Excelente calidad, superó mis expectativas.",              "positivo"),
    ("Estoy muy satisfecho con la compra, funciona perfecto.",   "positivo"),
    ("Muy buen precio y envío rápido. Volvería a comprar.",      "positivo"),
    # Negativos
    ("Terrible, se rompió al primer día de uso.",                "negativo"),
    ("No vale la pena el precio, muy decepcionante.",            "negativo"),
    ("El servicio al cliente es un desastre.",                   "negativo"),
    ("Llegó dañado y tardó 3 semanas.",                          "negativo"),
    # Neutros
    ("El producto llegó a tiempo.",                              "neutral"),
    ("Cumple su función básica.",                                "neutral"),
    ("Es lo que se describe en la página.",                      "neutral"),
    ("Ni muy bueno ni muy malo, promedio.",                      "neutral"),
]

print(f"Dataset: {len(DATASET)} ejemplos")
print(f"  Positivos: {sum(1 for _, l in DATASET if l == 'positivo')}")
print(f"  Negativos: {sum(1 for _, l in DATASET if l == 'negativo')}")
print(f"  Neutros:   {sum(1 for _, l in DATASET if l == 'neutral')}")


## 2. Definir los Prompts a Evaluar

In [ ]:
# Prompt v1: Simple
PROMPT_V1 = """
Clasifica el sentimiento del siguiente texto. 
Responde solo con una palabra: positivo, negativo, o neutral.

Texto: {text}
"""

# Prompt v2: Más detallado
PROMPT_V2 = """
Analiza el sentimiento del siguiente texto de reseña de producto.

Criterios:
- positivo: expresa satisfacción, recomendación o emociones positivas
- negativo: expresa insatisfacción, queja o emociones negativas  
- neutral: hechos objetivos sin carga emocional clara

Responde ÚNICAMENTE con una de estas palabras: positivo, negativo, neutral

Texto: {text}
"""

print("Prompts definidos: v1 (simple) y v2 (detallado)")

## 3. Ejecutar el Eval (Code-Based Grading)

In [ ]:
def run_eval(prompt_template: str, dataset: list, model: str = MODEL) -> dict:
    """Ejecuta el prompt sobre el dataset y calcula métricas."""
    results = []
    correct = 0
    total = len(dataset)

    for i, (text, expected) in enumerate(dataset, start=1):
        print(f"  Progreso: {i}/{total}")
        prompt = prompt_template.format(text=text)
        resp = client.messages.create(
            model=model, max_tokens=10,
            messages=[{"role": "user", "content": prompt}]
        )

        predicted = resp.content[0].text.strip().lower()
        # Normalizar predicción
        if "positivo" in predicted:
            predicted = "positivo"
        elif "negativo" in predicted:
            predicted = "negativo"
        else:
            predicted = "neutral"

        is_correct = predicted == expected
        if is_correct:
            correct += 1

        results.append({
            "text": text[:50],
            "expected": expected,
            "predicted": predicted,
            "correct": is_correct
        })

    accuracy = correct / len(dataset)
    return {"results": results, "accuracy": accuracy, "correct": correct, "total": len(dataset)}

print("Ejecutando Prompt V1...")
eval_v1 = run_eval(PROMPT_V1, DATASET)
print(f"V1 Accuracy: {eval_v1['accuracy']:.1%} ({eval_v1['correct']}/{eval_v1['total']})")

print("\nEjecutando Prompt V2...")
eval_v2 = run_eval(PROMPT_V2, DATASET)
print(f"V2 Accuracy: {eval_v2['accuracy']:.1%} ({eval_v2['correct']}/{eval_v2['total']})")


In [ ]:
# Analizar errores del mejor prompt
best_eval = eval_v2 if eval_v2['accuracy'] >= eval_v1['accuracy'] else eval_v1
best_name = "V2" if eval_v2['accuracy'] >= eval_v1['accuracy'] else "V1"

print(f"Analizando errores del Prompt {best_name}:")
errores = [r for r in best_eval['results'] if not r['correct']]

if not errores:
    print("Sin errores!")
else:
    for err in errores:
        print(f"\n  Texto: '{err['text']}'")
        print(f"  Esperado: {err['expected']} | Predicho: {err['predicted']}")


## 4. Model-Based Grading (LLM-as-Judge)

In [ ]:
JUDGE_SYSTEM = """
Eres un evaluador experto. Tu tarea es calificar respuestas generadas por un asistente de IA.
Debes ser estricto y objetivo. Responde ÚNICAMENTE con un JSON valido sin bloques de codigo:
{"score": <0-10>, "razon": "<breve explicación>"}
"""

def parse_json_safe(text: str) -> dict:
    """Parsea JSON aunque venga envuelto en bloques de código markdown."""
    text = text.strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
    return json.loads(text.strip())

def llm_judge(question: str, response: str, criteria: str) -> dict:
    """Usa un LLM para calificar la calidad de una respuesta."""
    prompt = f"""
Pregunta/Tarea: {question}

Respuesta a evaluar: {response}

Criterios de evaluación: {criteria}

Califica del 0 al 10.
"""
    resp = client.messages.create(
        model=MODEL_JUDGE, max_tokens=200,
        system=JUDGE_SYSTEM,
        messages=[{"role": "user", "content": prompt}]
    )
    return parse_json_safe(resp.content[0].text)

# Ejemplo: evaluar respuestas de resumen
texto_largo = """
La inteligencia artificial (IA) está transformando todos los sectores de la economía. 
Desde la medicina hasta la manufactura, las empresas están adoptando sistemas de IA 
para automatizar tareas, mejorar la precisión y reducir costos. Sin embargo, este 
avance también plantea preguntas importantes sobre el futuro del trabajo y la ética 
en el uso de estas tecnologías.
"""

resp_resumen = client.messages.create(
    model=MODEL, max_tokens=100,
    messages=[{"role": "user", "content": f"Resume en 1 oración: {texto_largo}"}]
)
resumen = resp_resumen.content[0].text
print("Resumen generado:")
print(resumen)

# Evaluar el resumen
evaluacion = llm_judge(
    question="Resumen en 1 oración del texto sobre IA",
    response=resumen,
    criteria="El resumen debe capturar las ideas principales, ser conciso (1 oración) y preciso"
)
print(f"\nEvaluacion del juez:")
print(f"  Score: {evaluacion['score']}/10")
print(f"  Razon: {evaluacion['razon']}")


## 5. Ejercicio: Eval Completo

**Tarea**: Crea un eval para comparar dos versiones de un prompt que genere preguntas de quiz.
El criterio: las preguntas deben ser claras, tener respuesta correcta y ser educativas.

In [ ]:
# Dataset de temas para generar preguntas
TEMAS = [
    "La Revolución Francesa",
    "El sistema solar",
    "La fotosíntesis",
    "La Segunda Guerra Mundial",
    "El ADN y la genética",
]

PROMPT_QUIZ_V1 = "Genera una pregunta de quiz sobre: {tema}"

PROMPT_QUIZ_V2 = """
Genera una pregunta de opción múltiple educativa sobre el tema: {tema}

Formato requerido:
PREGUNTA: [pregunta clara y específica]
A) [opción]
B) [opción]
C) [opción] 
D) [opción]
RESPUESTA_CORRECTA: [letra]
EXPLICACIÓN: [breve explicación de por qué es correcta]
"""

CRITERIOS_QUIZ = """
La pregunta debe:
1. Ser clara y sin ambigüedades
2. Tener exactamente 4 opciones (A, B, C, D)
3. Indicar claramente la respuesta correcta
4. Incluir una explicación educativa
Penaliza fuertemente si falta cualquiera de estos elementos.
"""

scores_v1, scores_v2 = [], []

total_temas = len(TEMAS)
for i, tema in enumerate(TEMAS, start=1):
    print(f"\nTema {i}/{total_temas}: {tema}")
    # V1
    r1 = client.messages.create(
        model=MODEL, max_tokens=300,
        messages=[{"role": "user", "content": PROMPT_QUIZ_V1.format(tema=tema)}]
    ).content[0].text
    j1 = llm_judge(f"Pregunta de quiz sobre {tema}", r1, CRITERIOS_QUIZ)
    scores_v1.append(j1['score'])

    # V2
    r2 = client.messages.create(
        model=MODEL, max_tokens=300,
        messages=[{"role": "user", "content": PROMPT_QUIZ_V2.format(tema=tema)}]
    ).content[0].text
    j2 = llm_judge(f"Pregunta de quiz sobre {tema}", r2, CRITERIOS_QUIZ)
    scores_v2.append(j2['score'])

    print(f"  {tema}: V1={j1['score']}/10  V2={j2['score']}/10")

print(f"\nResultados finales:")
print(f"  Prompt V1 promedio: {sum(scores_v1)/len(scores_v1):.1f}/10")
print(f"  Prompt V2 promedio: {sum(scores_v2)/len(scores_v2):.1f}/10")
ganador = "V2" if sum(scores_v2) > sum(scores_v1) else "V1"
print(f"  Ganador: Prompt {ganador}")
